In [10]:
#!pip install langchain langchain-core langchain_community langchain_openai
#Using langchain for templates
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate

In [11]:
#Check versions
import os
import transformers
import numpy as np
import tokenizers
import sys
import transformers.models.t5
import langchain_huggingface
print("transformers",transformers.__version__)
print("numpy",np.__version__)
print(transformers.__file__)
print(transformers.models.t5.__file__)
print("Python:", sys.executable)
print("Transformers path:", transformers.__file__)
print("Tokenizers:", tokenizers.__version__)
print("Tokenizers path:", tokenizers.__file__)
from importlib.metadata import version
print(version("langchain-huggingface"))
!pip show torch
!pip show torchvision
!pip show torchaudio

transformers 4.55.2
numpy 1.26.4
e:\Lesson_2_demos\venv\lib\site-packages\transformers\__init__.py
e:\Lesson_2_demos\venv\lib\site-packages\transformers\models\t5\__init__.py
Python: e:\Lesson_2_demos\venv\Scripts\python.exe
Transformers path: e:\Lesson_2_demos\venv\lib\site-packages\transformers\__init__.py
Tokenizers: 0.21.4
Tokenizers path: e:\Lesson_2_demos\venv\lib\site-packages\tokenizers\__init__.py
1.2.2
Name: torch
Version: 2.2.2
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: e:\lesson_2_demos\venv\lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, sympy, typing-extensions
Required-by: accelerate, sentence-transformers, torchvision
Name: torchvision
Version: 0.17.2
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
Author-email: 

In [2]:
#Using different models 
#(may throw error due to tokenizer being new version which came in when we installed other packages)
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

In [4]:
#downgrade protobuf to avoid warnings,if needed
#!pip install --upgrade protobuf==4.25.3
#restart session,if above step done
!pip show protobuf

Name: protobuf
Version: 7.34.0
Summary: 
Home-page: https://developers.google.com/protocol-buffers/
Author: protobuf@googlegroups.com
Author-email: protobuf@googlegroups.com
License: 3-Clause BSD License
Location: e:\lesson_2_demos\venv\lib\site-packages
Requires: 
Required-by: googleapis-common-protos, onnxruntime, opentelemetry-proto, streamlit, tensorboard, tensorflow


In [3]:
#Using smaller model than xl
#Online Mode
#generator = pipeline("text2text-generation", model="google/flan-t5-large")

#Offline Mode (to avoid redownloading or checking latest on web)
model_name = "google/flan-t5-large"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
)

def get_completion(prompt):
    response = generator(prompt,max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())

print(get_completion("What is a DEFI in context of crypto world"))
#Using different variant i.e larger model, to run remove """ """

Device set to use cpu


decentralized exchange for fiat money


In [4]:
#Back to Chain Approach & using templates

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

Device set to use cpu


In [7]:
prompt = "What is crypto currency"
response = generator(prompt)
print(response)

[{'generated_text': 'crypto currency'}]


In [9]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline

template2 = "Please write a {length} review,of the book {book_title}. "
#template2 = "Summarize the book {book_title} in {length} words. "
input_variables2 = [ "length", "book_title" ]
prompt = PromptTemplate(
    input_variables=input_variables2,
    template=template2
)
#To check
#formatted_prompt = prompt.format(length = "short", book_title = " House Of Dragon")
#print(formatted_prompt)
llm = HuggingFacePipeline(pipeline=generator)

#Chaining
chain_new = prompt | llm
response1 = chain_new.invoke({
    "length": "short",
    "book_title": "House of Dragon"
})
print('response1-chain: ',response1)

#Passing prompt directly into Pipeline
prompt = template2.format(
    length="short",
    book_title="House of Dragon"
)
response2 = generator(prompt)
print('response2-direct: ', response2[0]["generated_text"])

#Passing prompt into Function
prompt = template2.format(
    length="short",
    book_title="Sherlock Holmes"
)
response3 = get_completion(prompt)
print('response3-func: ',response3)


response1-chain:  The book House of Dragon is a story about a young girl who lives in a house with a dragon. The dragon is a young girl who lives in a house with a dragon. The dragon is a young girl who lives in a house with a dragon. The book House of Dragon is a story about a young girl who lives in a house with a dragon. The dragon is a young girl who lives in a house with
response2-direct:  The book House of Dragon is a story about a young girl who lives in a house with a dragon. The dragon is a young girl who lives in a house with a dragon. The dragon is a young girl who lives in a house with a dragon. The book House of Dragon is a story about a young girl who lives in a house with a dragon. The dragon is a young girl who lives in a house with
response3-func:  Sherlock Holmes is one of the best novels ever written. It's also one of the best films ever made. It's also one of the best films ever made. It's also one of the best films ever made. It's also one of the best films ever ma

In [ ]:
#Using function as defined above
formatted_prompt = prompt.format(length = "short", book_title = " House Of Dragon")
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response)

AI Response:
<class 'str'>
Sherlock Holmes is a witty, witty, witty novel about a man who is a master of his craft . . . and a master of the universe . . . and a master of the universe . . . and a master of the universe . . . and a master of the universe . . . and a master of the universe . . .


#### Using Other Models
- Mistral

In [2]:
#from transformers import pipeline
#Note**Access to model mistralai/Mistral-7B-Instruct-v0.1 is restricted. You must have access to it and be
#authenticated to access it. If yes, then we can use the code below.
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run
"""
generator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.1")


def get_completion(prompt):
    # For instruction-tuned models, prepend with an instruction-style format
    instruction = f"<s>[INST] {prompt} [/INST]"
    response = generator(instruction, max_new_tokens=100, do_sample=False)
    return response[0]["generated_text"].split("[/INST]")[-1].strip()

print(get_completion("What is defi in context of crypto world?"))
"""

'\ngenerator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.1")\n\n\ndef get_completion(prompt):\n    # For instruction-tuned models, prepend with an instruction-style format\n    instruction = f"<s>[INST] {prompt} [/INST]"\n    response = generator(instruction, max_new_tokens=100, do_sample=False)\n    return response[0]["generated_text"].split("[/INST]")[-1].strip()\n\nprint(get_completion("What is defi in context of crypto world?"))\n'

- Falcon

In [ ]:
##using Another heavier model
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run
"""
# Load a text-generation pipeline with an instruction-tuned model
generator = pipeline("text-generation", model="tiiuae/falcon-7b-instruct")

def get_completion(prompt):
    # For instruction-tuned models, prepend with an instruction-style format
    instruction = f"<s>[INST] {prompt} [/INST]"
    response = generator(instruction, max_new_tokens=100, do_sample=False)
    return response[0]["generated_text"].split("[/INST]")[-1].strip()

print(get_completion("What is defi in context of crypto world?"))
"""

##### Offline/Online mode - custom location for models
###### Download Model to a Custom Location (D: Drive)

In [4]:
#Set the TRANSFORMERS_CACHE environment variable before importing transformers
#Ignoring these for now,as we are doing this in cells below
#import os
#os.environ["TRANSFORMERS_CACHE"] = "D:\\huggingface"   # Windows
# os.environ["TRANSFORMERS_CACHE"] = "/d/huggingface" # Git Bash / WSL
#import torch
#from transformers import pipeline

In [5]:
#Alternatively, pass cache_dir directly in the pipeline call (more explicit and portable but sometimes depends on models):
#Both approaches work — cache_dir in the call overrides the env variable if both are set.
# generator = pipeline(
#     "text-generation",
#     model="tiiuae/falcon-7b-instruct",
#     cache_dir="D:\\huggingface_models"
# )

In [6]:
#So we can..
'''
Falcon-7b-instruct is ~14GB on disk and needs ~16GB RAM/VRAM. 
If you're on CPU only, generation will be very slow — consider tiiuae/falcon-rw-1b for testing.
See example below
'''
# generator = pipeline(
#     "text-generation",
#     model="tiiuae/falcon-7b-instruct",
#     torch_dtype=torch.bfloat16,         # saves memory; use float32 if issues arise
#     trust_remote_code=True,             # required for Falcon
#     device_map="auto",                  # auto GPU/CPU placement
#     cache_dir="D:\\huggingface_models"   # explicit, overrides env var too
# )

"\nFalcon-7b-instruct is ~14GB on disk and needs ~16GB RAM/VRAM. \nIf you're on CPU only, generation will be very slow — consider tiiuae/falcon-rw-1b for testing.\nSee example below\n"

In [7]:
# If downloaded then Test it
# def get_completion(prompt, max_new_tokens=200):
#     instruction = f"User: {prompt}\nFalcon:"   # Falcon instruct format
#     response = generator(
#         instruction,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         eos_token_id=generator.tokenizer.eos_token_id,
#         pad_token_id=generator.tokenizer.eos_token_id  # avoids padding warning
#     )
#     generated = response[0]["generated_text"]
#     # Strip the prompt prefix, return only the model's reply
#     return generated[len(instruction):].strip()

# print(get_completion("What is DeFi in the context of the crypto world?"))

In [8]:
#consider tiiuae/falcon-rw-1b
#falcon-rw-1b: a base model, not instruct-tuned, so we need to adjust the prompt format — 
#drop the User:/Falcon: wrapper and just pass the prompt directly:
'''
Download is only ~2.5GB, much faster
Since it's a base model, answers will be more like text continuation than clean Q&A responses
'''
import os
cache = r"D:\\huggingface"

os.environ["HF_HOME"] = cache
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(cache, "hub")
os.environ["HF_MODULES_CACHE"] = os.path.join(cache, "modules")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(cache, "transformers")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [9]:
#Check paths
import os

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_MODULES_CACHE:", os.environ.get("HF_MODULES_CACHE"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

HF_HOME: D:\\huggingface
HF_MODULES_CACHE: D:\\huggingface\modules
TRANSFORMERS_CACHE: D:\\huggingface\transformers


In [10]:
#To check if notebook or virtual environment already has cache locations configured
import os
import transformers
import huggingface_hub

print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

for key in [
    "HF_HOME",
    "HUGGINGFACE_HUB_CACHE",
    "HF_MODULES_CACHE",
    "TRANSFORMERS_CACHE",
    "XDG_CACHE_HOME",
]:
    print(f"{key} =", os.environ.get(key))

e:\Lesson_2_demos\venv\lib\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


transformers: 4.55.2
huggingface_hub: 0.34.4
HF_HOME = D:\\huggingface
HUGGINGFACE_HUB_CACHE = D:\\huggingface\hub
HF_MODULES_CACHE = D:\\huggingface\modules
TRANSFORMERS_CACHE = D:\\huggingface\transformers
XDG_CACHE_HOME = None


In [ ]:
#If model downloaded (& Note: transformers is lower than 4.55 such as 4.37, then we can test this)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained(
    "tiiuae/falcon-rw-1b",
    #cache_dir=cache,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-rw-1b",
    #cache_dir=cache,
    trust_remote_code=True,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def get_completion(prompt, max_new_tokens=200):
    response = generator(
        prompt,                          # no instruction wrapper needed
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )
    generated = response[0]["generated_text"]
    return generated[len(prompt):].strip()   # strip the input prompt from output

In [11]:
#Check
from pathlib import Path
root = Path(r"D:\\huggingface")
for p in root.rglob("model.safetensors"):
    print(p)

D:\huggingface\transformers\models--tiiuae--falcon-rw-1b\.no_exist\e4b9872bb803165eb22f0a867d4e6a64d34fce19\model.safetensors


In [ ]:
#If all good and downloaded
print(get_completion("What is DeFi in the context of the crypto world?"))
#If errors, try next cell too

In [ ]:
#Testing compatibility of falcon with transformers
#This may take lot of time and still not work as falcon with transformers-4.55 may have compatibiltiy issue
#so try it and if it takes time, interrupt kernel

import torch

prompt = "What is DeFi in the context of the crypto world?"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=False,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [12]:
from huggingface_hub import scan_cache_dir
info = scan_cache_dir(r"D:\huggingface\transformers")
print(info)
#might show: corruption error

HFCacheInfo(size_on_disk=0, repos=frozenset(), warnings=[CorruptedCacheException("Reference(s) refer to missing commit hashes: {'79322c51ba0241c81368bdeb9c6795398aeeb22e': {'refs\\\\pr\\\\7'}} (D:\\huggingface\\transformers\\models--tiiuae--falcon-rw-1b).")])


### Switching to usage of bigger LLMs deployed on endpoints

In [ ]:
#repeating previous steps
template2 = "Please write a {length} review,of the book {book_title}. "
input_variables2 = [ "length", "book_title" ]

prompt = template2.format(
    length="short",
    book_title="House of Dragon"
)

#Using AzureOpenAI and GPT model
#Note** If using gpt model and AzureOpenAI or AzureChatOpenAI (refer: 'Working_with_AzureOpenAI' files)
import openai
import os
from openai import AzureOpenAI

# Initialize client once
from dotenv import load_dotenv
#load_dotenv("/content/.env")
load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-12-01-preview",
)
deployment_name = os.getenv("AZURE_DEPLOYMENT_NAME")
#or


#Using Langchain Equivalent (AzureOpenAI or AzureChatOpenAI)
'''
from langchain_openai import AzureChatOpenAI

#from dotenv import load_dotenv
#load_dotenv("/content/.env")
#load_dotenv()

client = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("API_KEY"),
    api_version="2024-12-01-preview",
    deployment_name="gpt-4.1",
    temperature=0,
)

client.invoke("Explain transformers in simple terms in 25 words").content
'''

#Using OpenAI
#When using newer version (if openai upgraded)
# Import libraries
'''
from openai import OpenAI

openai_base_url = os.getenv("OPENAI_BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")
model="gpt-3.5-turbo"

# Initialize OpenAI client
client = OpenAI(
    base_url=openai_base_url,
    api_key=openai_api_key
)

print("Environment setup complete.")

response = client.chat.completions.create(
    model=model,          # or "gpt-5", "gpt-4o", etc.
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)
'''

'\nfrom openai import OpenAI\n\nopenai_base_url = os.getenv("OPENAI_BASE_URL")\nopenai_api_key = os.getenv("OPENAI_API_KEY")\n\n# Initialize OpenAI client\nclient = OpenAI(\n    base_url=openai_base_url,\n    api_key=openai_api_key\n)\n\nprint("Environment setup complete.")\n\nresponse = client.chat.completions.create(\n    model="gpt-3.5-turbo",          # or "gpt-5", "gpt-4o", etc.\n    messages=[\n        {\n            "role": "user",\n            "content": prompt\n        }\n    ],\n    temperature=0\n)\n\nprint(response.choices[0].message.content)\n'

In [6]:
#Creating function (using Client based on AzureOpenAI or AzureChatOpenAI)
def get_completion(prompt, deployment_name=deployment_name):
    """Get a chat completion from Azure OpenAI.
    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.
    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )
        #return response.model_dump()  # Return the full response as dict
        # Extract just the assistant's reply
        return response.choices[0].message.content
    except Exception as e:
        return {"error": str(e)}

In [3]:
# def get_realtime_data(prompt):
#     print('realtime weather is good')

prompt = "what is the cryptography"

# if 'now' in prompt:
#     get_realtime_data(prompt)
# else:
get_completion(prompt)



'**Cryptography** is the science and art of securing information by transforming it into a form that only intended recipients can understand. It involves techniques for **encrypting** (scrambling) data to protect it from unauthorized access and **decrypting** (unscrambling) it for authorized users.\n\n### Key Concepts in Cryptography\n\n- **Encryption:** The process of converting plain text (readable data) into ciphertext (unreadable, scrambled data) using an algorithm and a key.\n- **Decryption:** The reverse process, converting ciphertext back into plain text using a key.\n- **Cipher:** The algorithm used for encryption and decryption.\n- **Key:** A secret value used by the cipher to encrypt and decrypt data.\n- **Symmetric Cryptography:** The same key is used for both encryption and decryption (e.g., AES, DES).\n- **Asymmetric Cryptography:** Uses a pair of keys—public and private—for encryption and decryption (e.g., RSA, ECC).\n\n### Uses of Cryptography\n\n- **Confidentiality:** E

In [ ]:
#If using get_completion() based on gpt model as defined above, then we can
"""
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response.keys())

response['choices'][0]['message']['content']"""

In [ ]:
#When Using OpenAI
'''
def get_completion(prompt, model="gpt-4.1"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    #return response.choices[0].message.content
    return response

# Test call
print(get_completion(prompt))
'''

ChatCompletion(id='chatcmpl-E30X1ZKaoHe4NyHqvYTIx8mJtvgif', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**Cryptography** is the science and art of securing information by transforming it into a form that only intended recipients can understand. It involves techniques for **encrypting** (scrambling) data to protect it from unauthorized access and **decrypting** (unscrambling) it so authorized users can read it.\n\n### Key Concepts in Cryptography\n\n- **Encryption:** The process of converting plain text (readable data) into ciphertext (unreadable data) using an algorithm and a key.\n- **Decryption:** The process of converting ciphertext back into plain text using a key.\n- **Key:** A piece of information (a string of bits) used by an algorithm to encrypt and decrypt data.\n- **Cipher:** The algorithm used for encryption and decryption.\n- **Symmetric Cryptography:** The same key is used for both encryption and decryption (e.g., AE

#### Formatting
- ##### Jinja template

In [13]:
from langchain_core.prompts import PromptTemplate
#Jinja Template example
jinja2_template = "Give me an {{ adjective }} fact about {{ topic }}"

In [14]:
prompt = PromptTemplate.from_template(jinja2_template, template_format = "jinja2" )

In [15]:
user_question = prompt.format(adjective="interesting", topic="space exploration")
print(user_question)

Give me an interesting fact about space exploration


In [22]:
response = get_completion(user_question)
#print("AI Response:")
print(response)
#print(type(response))

Sure! Did you know that **Voyager 1**, launched by NASA in 1977, is the farthest human-made object from Earth? As of now, it's over 14 billion miles (23 billion kilometers) away and has entered interstellar space, sending data back to Earth from beyond our solar system!


- ##### F-string

In [27]:
#Using f-string (example)
fstring_template = "Give me a brief summary for the book titled '{book_title}':"
book_title = "The Great Gatsby"
prompt = fstring_template.format(book_title=book_title)

In [24]:
#Testing a dummy function
def get_book_summary(prompt):
    return "It's a novel about love, wealth, and aspiration, set in the Roaring '20s."

In [25]:
summary = get_book_summary(prompt)
print(summary)

It's a novel about love, wealth, and aspiration, set in the Roaring '20s.


In [28]:
#Using function that invokes the LLM
response = get_completion(prompt)
print("AI Response:")
print(response)

AI Response:
**The Great Gatsby** by F. Scott Fitzgerald is a classic novel set in the 1920s on Long Island, New York. The story is narrated by Nick Carraway, who becomes entangled in the lavish and mysterious world of his wealthy neighbor, Jay Gatsby. Gatsby is famous for his extravagant parties and harbors an obsessive love for Daisy Buchanan, Nick’s cousin, who is married to the wealthy but unfaithful Tom Buchanan. The novel explores themes of the American Dream, wealth, love, and social class, ultimately revealing the emptiness and moral decay beneath the glittering surface of the Jazz Age.


In [29]:
# Example where string prompt template would not work
# Define the prompt template

jinja2_template = """

Dear {{ name }},
{% if age < 18 %}
You are invited to our kids' event with activities such as face painting, bouncy castles, and clown shows.
{% elif age < 65 %}
You are invited to our adult event with activities like live music, wine tasting, and art workshops.
{% else %}
You are invited to our senior event with activities including book clubs, chess tournaments, and tea dances.
{% endif %}
Sincerely,
Event Organizer

Write the mail in 200 words
"""

In [33]:
prompt = PromptTemplate.from_template(jinja2_template, template_format="jinja2")

# Format the prompt with specific values for 'action', 'group', and 'time_period'
argument_prompt = prompt.format(name="John Doe", age=55)

In [31]:
response = get_completion(argument_prompt)
print("AI Response:")
print(response)

AI Response:
Subject: Invitation to Our Exclusive Adult Event

Dear John Doe,

We are delighted to invite you to an exclusive adult event designed for an evening of enjoyment, creativity, and connection. Join us for a memorable night featuring live music performances, curated wine tasting sessions, and engaging art workshops led by talented local artists.

This event is the perfect opportunity to unwind, socialize, and explore new interests in a vibrant atmosphere. Whether you are a wine enthusiast, an art lover, or simply looking for a unique night out, there will be something for everyone to enjoy. Our live music lineup promises to set the perfect mood, while the wine tasting will introduce you to a selection of exquisite local and international wines. The art workshops are suitable for all skill levels, so feel free to express your creativity and take home your own masterpiece.

The event will take place on [Date] at [Venue], starting at [Time]. Dress code is smart casual. Please RS

In [ ]:
# #print(argument_prompt,'\n',
# #user_question)

# choices = {1: argument_prompt,2: user_question}

# user_choice = input('choose a prompt option from above')
# if user_choice == 1:
#     get_completion(prompt=choices[1])
# # else:
#     get_completion(prompt=choices[2])

#### Using Langchain templates & chat mode

In [54]:
#Define raw template strings
#Just plain Python strings with {placeholders} — nothing LangChain-specific.
simple_prompt = "The {subject} is strong in this one."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [55]:
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
'''
HumanMessage / AIMessage — represent a single message with a role attached (who said it)
HumanMessagePromptTemplate — wraps a template string and stamps it as coming from the "human" role
ChatPromptTemplate — assembles multiple message templates into a full conversation structure
MessagesPlaceholder — a slot where you inject a list of existing messages (e.g. prior chat history)

'''

'\nHumanMessage / AIMessage — represent a single message with a role attached (who said it)\nHumanMessagePromptTemplate — wraps a template string and stamps it as coming from the "human" role\nChatPromptTemplate — assembles multiple message templates into a full conversation structure\nMessagesPlaceholder — a slot where you inject a list of existing messages (e.g. prior chat history)\n\n'

In [56]:
#Build the chat prompt structure
'''
This defines a conversation blueprint with three slots in order:

MessagesPlaceholder — inject existing chat history here
A human message: "The {subject} is strong in this one."
A human message: "Summarize our conversation so far in {word_count} words."
'''
simple_message_template = HumanMessagePromptTemplate.from_template(simple_prompt)
human_message_template = HumanMessagePromptTemplate.from_template(human_prompt)

chat_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="conversation"),
    simple_message_template,
    human_message_template
])

In [57]:
human_message = HumanMessage(content="What's the best way to learn a new language?")
ai_message = AIMessage(content="""\
1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.
2. Practice regularly: Consistency is key when learning a new language.
3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.\
""")

In [58]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Force",
    word_count="10"
).to_messages()
'''All placeholders filled in. .to_messages() converts the whole thing into a flat list of message objects ready 
to send to a model. print(conversation) lets you see the final assembled structure. '''

'All placeholders filled in. .to_messages() converts the whole thing into a flat list of message objects ready \nto send to a model. print(conversation) lets you see the final assembled structure. '

In [59]:
print(conversation)

[HumanMessage(content="What's the best way to learn a new language?", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.\n2. Practice regularly: Consistency is key when learning a new language.\n3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='The Force is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [60]:
#Demonstrates that the same chat_prompt template is reusable — you just pass different values each time.
simple_prompt = "The {subject} is fascinating to study."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [61]:
human_message = HumanMessage(content="What's happens inside a black hole")
ai_message = AIMessage(content="""\
1. Inside black hole gravity is zero way.\
""")

In [62]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Black Hole",
    word_count="10"
).to_messages()

In [63]:
prompt_string = conversation
print("Formatted Prompt:")
print(prompt_string)


Formatted Prompt:
[HumanMessage(content="What's happens inside a black hole", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Inside black hole gravity is zero way.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='The Black Hole is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [41]:
print(type(prompt_string))

<class 'list'>


In [64]:
# Convert LangChain message objects to a plain string for get_completion
# Each message has a .content attribute; join all message contents
'''
prompt_string is a list of message objects — get_completion needs a plain string. 
So we extract .content from each message and join them with newlines before passing to the flan-t5 pipeline/further. 
'''
prompt_string_text = "\n".join(msg.content for msg in prompt_string)

response = get_completion(prompt_string_text)
print("AI Response:")
print(response)

AI Response:
Black hole: gravity strong, not zero; discussed black hole interior.


In [ ]:
#Using Langchain templates & styles

In [7]:
#Using style
#use a prompt template to control the style/tone of model output 
#Modify parameters like **customer_style** and **customer_email**
#to influence the tone and formality of the generated responses.

"""    
Two variables: {style} controls tone, {text} is the content to restyle. Triple backticks are used to clearly 
delimit the input text from the instruction — a common prompting pattern.
"""

template_string = """Translate the text that is delimited by triple backticks into a style
that is {style}. text: ```{text}```"""

# Style and email input
customer_style = "American English in a casual tone"
customer_email = """
I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!
"""

In [8]:
prompt = template_string.format(style=customer_style, text=customer_email)

In [9]:
#instruction_prompt = f"<s>[INST] {prompt} [/INST]"
# [INST] format is for Mistral/Llama models only; flan-t5 takes plain text
instruction_prompt = prompt  # pass prompt directly, no [INST] wrapper needed

In [47]:
#Using generator based on google/flan-t5-large
#Using generator based on google/flan-t5-large
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

generator = pipeline(
    "text2text-generation",
    #model=model,
    #tokenizer=tokenizer,
    max_new_tokens=100, 
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

response = generator(prompt, max_new_tokens=150, do_sample=False)

No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [48]:
response

[{'generated_text': 'True'}]

In [49]:
generated_text = response[0]['generated_text'].split("[/INST]")[-1].strip()

In [50]:
print(response[0])

{'generated_text': 'True'}


In [10]:
#using our function
response = get_completion(prompt)
print(response)

I'm so pumped about the new gaming console I got! It showed up in just two days, and I've been playing nonstop ever since. Totally worth the money!


In [52]:
response = get_completion(instruction_prompt)
print(response)

I'm really pumped about the new gaming console I got! It showed up in just two days, and I've been playing nonstop ever since. Totally worth the money!


In [11]:
#to be checked...
template_string = """Translate the text that is delimited by triple backticks into a style that is {style}.
text: ```{text}```"""

'''
prompt_template = ChatPromptTemplate.from_template(template_string)
message = prompt_template.format_messages(
    style="Scottish English in a professional tone",
    text="I'm super excited about the new gaming console & new game!"
)'''

style="Scottish english in a professional tone"
text="I'm super excited about the new gaming console & new game!"


prompt = template_string.format(style=style, text=text)

#Using generator based on google/flan-t5-base
#response = generator(prompt,max_new_tokens=150, do_sample=False)
response = get_completion(prompt)
print(response)

I’m absolutely delighted about the new gaming console and the latest game!
